# LeetCode 2131 — Longest Palindrome by Concatenating Two Letter Words bằng Python

Link đề: https://leetcode.com/problems/longest-palindrome-by-concatenating-two-letter-words/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, String, Greedy, Counting

Skeleton LeetCode:

```python
class Solution:
    def longestPalindrome(self, words: List[str]) -> int:
        ...
```

## 1. Mục tiêu học

1. Phân tích **cấu trúc** của một palindrome ghép từ các khối 2 chữ: khối thứ `i` từ trái phải là **đảo ngược** của khối thứ `i` từ phải; khối giữa (nếu có) phải tự đối xứng (`"aa"`).
2. Phân loại từ thành hai nhóm: **từ đối xứng** (`"gg"`) và **từ không đối xứng** (`"lc"` ghép với `"cl"`).
3. Dùng `Counter` để **ghép cặp theo tần suất** thay vì thử thứ tự — từ O(n!) xuống O(n).
4. Cài 4 cách: vét cạn tập con + hoán vị, ghép cặp tham lam một lượt, `Counter` (cách doocs), mảng 26×26.
5. Xử lý tinh tế phần **"tâm"**: chỉ được đặt **một** từ đối xứng lẻ ở giữa.

## 2. Đề bài & ví dụ

Cho mảng chuỗi `words`, mỗi chuỗi gồm **đúng 2 chữ cái thường**. Chọn một số phần tử (mỗi phần tử dùng **tối đa một lần**) và ghép theo **thứ tự tùy ý** để tạo palindrome **dài nhất**. Trả về độ dài đó (0 nếu không tạo được).

**Ví dụ 1:** `["lc","cl","gg"]` → `6` (`"lc"+"gg"+"cl" = "lcggcl"`).

**Ví dụ 2:** `["ab","ty","yt","lc","cl","ab"]` → `8` (`"tylcclyt"`; hai `"ab"` không dùng được vì không có `"ba"`).

**Ví dụ 3:** `["cc","ll","xx"]` → `2` (chỉ đặt **một** từ đối xứng ở giữa).

**Ràng buộc:** `1 ≤ words.length ≤ 10⁵`, `words[i].length == 2`, chữ cái thường.

In [1]:
from typing import List
from collections import Counter
from itertools import permutations, combinations
import random, time

examples = [
    (["lc", "cl", "gg"], 6),
    (["ab", "ty", "yt", "lc", "cl", "ab"], 8),
    (["cc", "ll", "xx"], 2),
]
def is_pal(s):
    return s == s[::-1]
for w, exp in examples:
    print(w, "-> kỳ vọng", exp)
print(is_pal("lcggcl"), is_pal("tylcclyt"), is_pal("lcgg"))

['lc', 'cl', 'gg'] -> kỳ vọng 6
['ab', 'ty', 'yt', 'lc', 'cl', 'ab'] -> kỳ vọng 8
['cc', 'll', 'xx'] -> kỳ vọng 2
True True False


## 3. Kiến thức nền

### 3.1 Đảo chuỗi bằng slicing `s[::-1]` và kiểm tra palindrome

In [2]:
w = "lc"
print(w[::-1], w[::-1] == "cl", "gg"[::-1] == "gg")
print(w[0] == w[1], "gg"[0] == "gg"[1])     # từ 2 chữ đối xứng <=> hai chữ bằng nhau

cl True True
False True


### 3.2 `collections.Counter` — đếm tần suất

`Counter(words)` cho `dict` từ → số lần xuất hiện. Truy cập khóa không có trả về `0` (không `KeyError`).

In [3]:
cnt = Counter(["ab", "ty", "yt", "lc", "cl", "ab"])
print(cnt)
print(cnt["ab"], cnt["ba"], "ba" in cnt)   # cnt["ba"] = 0 nhưng KHÔNG tạo khóa mới

Counter({'ab': 2, 'ty': 1, 'yt': 1, 'lc': 1, 'cl': 1})
2 0 False


### 3.3 Phép chia nguyên và chẵn lẻ: `v // 2`, `v % 2`, `v & 1`

Với `v` bản của một từ đối xứng, ghép được `v // 2` cặp; còn dư `v % 2` (0 hoặc 1) bản. `v & 1` là cách viết bit của `v % 2`.

In [4]:
for v in [0, 1, 4, 5]:
    print(f"v={v}: cặp={v // 2}, dư={v % 2}, v&1={v & 1}")

v=0: cặp=0, dư=0, v&1=0
v=1: cặp=0, dư=1, v&1=1
v=4: cặp=2, dư=0, v&1=0
v=5: cặp=2, dư=1, v&1=1


### 3.4 Ánh xạ chữ cái → số: `ord(c) - ord('a')`

Dùng cho cách mảng 26×26: từ `"xy"` ứng với ô `[x][y]`, từ đảo `"yx"` là ô `[y][x]`.

In [5]:
def idx(c): return ord(c) - ord("a")
print(idx("a"), idx("l"), idx("c"), "-> 'lc' ở ô", (idx("l"), idx("c")), ", 'cl' ở ô", (idx("c"), idx("l")))

0 11 2 -> 'lc' ở ô (11, 2) , 'cl' ở ô (2, 11)


## 4. Ý tưởng nền tảng / trực giác

Một palindrome ghép từ `k` khối 2 chữ: `B_1 B_2 … B_k`. Vì mọi khối dài đúng 2, ranh giới các khối **đối xứng** qua tâm chuỗi. Đọc ngược chuỗi:
- `B_1` (ký tự 1–2) phải khớp với **đảo ngược** của `B_k` (hai ký tự cuối).
- Tổng quát: `B_i = reverse(B_{k+1−i})`.
- Nếu `k` lẻ, khối giữa `B_{(k+1)/2}` phải bằng đảo ngược của chính nó ⇒ là từ dạng `"aa"`.

Ví dụ 2: `ty | lc | cl | yt` — `ty ↔ yt`, `lc ↔ cl`. Hai `"ab"` không có `"ba"` ⇒ bỏ.

Vậy bài toán thành **ghép cặp**:
- Từ không đối xứng `"xy"` ghép với `"yx"`: mỗi cặp đóng góp **4** ký tự (đặt đối xứng hai bên).
- Từ đối xứng `"aa"` ghép với một `"aa"` khác: mỗi cặp đóng góp **4** ký tự.
- Sau khi ghép xong, nếu còn **ít nhất một** từ đối xứng lẻ ⇒ đặt nó ở giữa, **+2** (chỉ một lần!).

Tính tay ví dụ 3 `["cc","ll","xx"]`: không có cặp nào; còn dư 3 từ đối xứng ⇒ chỉ đặt 1 ở giữa ⇒ 2.

In [6]:
def explain(words):
    cnt = Counter(words); pairs = []; center = []
    seen = set()
    for w, v in cnt.items():
        if w[0] == w[1]:
            pairs += [(w, w)] * (v // 2)
            if v % 2: center.append(w)
        elif w not in seen:
            r = w[::-1]; k = min(v, cnt[r])
            pairs += [(w, r)] * k
            seen.add(w); seen.add(r)
    left = "".join(a for a, _ in pairs); right = "".join(b for _, b in reversed(pairs))
    mid = center[0] if center else ""
    s = left + mid + right
    print(f"{words}\n   cặp: {pairs}, ứng viên tâm: {center}\n   palindrome: '{s}' (dài {len(s)}), kiểm tra: {is_pal(s)}")
for w, _ in examples:
    explain(w)

['lc', 'cl', 'gg']
   cặp: [('lc', 'cl')], ứng viên tâm: ['gg']
   palindrome: 'lcggcl' (dài 6), kiểm tra: True
['ab', 'ty', 'yt', 'lc', 'cl', 'ab']
   cặp: [('ty', 'yt'), ('lc', 'cl')], ứng viên tâm: []
   palindrome: 'tylcclyt' (dài 8), kiểm tra: True
['cc', 'll', 'xx']
   cặp: [], ứng viên tâm: ['cc', 'll', 'xx']
   palindrome: 'cc' (dài 2), kiểm tra: True


## 5. Cách 1 — Brute Force / baseline: thử mọi tập con và mọi thứ tự

**Ý tưởng:** đúng như đề: chọn một tập con, thử mọi hoán vị, nếu chuỗi ghép là palindrome thì cập nhật độ dài. Thử từ tập con lớn xuống để dừng sớm.

In [7]:
def brute(words):
    n = len(words)
    for k in range(n, 0, -1):                       # từ nhiều từ tới ít từ
        for comb in combinations(range(n), k):
            for perm in permutations(comb):
                s = "".join(words[i] for i in perm)
                if s == s[::-1]:
                    return 2 * k
    return 0

for w, exp in examples:
    got = brute(w)
    print(got, "OK" if got == exp else "SAI")

6 OK
8 OK
2 OK


**Độ phức tạp:** Σ C(n,k)·k! ≈ e·n! tổ hợp, mỗi cái O(n) ⇒ **O(n!·n)**. n = 8 đã ~10⁵ hoán vị; n = 10⁵ là không tưởng.

**Nút thắt:** thứ tự ghép chỉ cần tuân theo cấu trúc "đối xứng qua tâm" ở mục 4 — không cần thử. Điều quan trọng duy nhất là **mỗi loại từ xuất hiện bao nhiêu lần**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** độ dài palindrome lớn nhất = `4 × (số cặp ghép được) + 2 × [còn từ đối xứng chưa ghép]`.

Với bảng đếm `cnt`:
- **Nếu** `w = "xy"` với `x ≠ y` **thì** số cặp `("xy", "yx")` = `min(cnt["xy"], cnt["yx"])` (mỗi bản `"xy"` cần một bản `"yx"` riêng).
- **Nếu** `w = "xx"` **thì** số cặp = `cnt["xx"] // 2`, dư `cnt["xx"] % 2`.
- **Nếu** có bất kỳ `"xx"` nào dư **thì** cộng thêm 2 (một từ ở tâm) — **chỉ một lần** cho cả bài.

**Vì sao tham lam ghép tối đa là tối ưu?** Các loại từ không tranh chấp nhau: `"xy"` chỉ ghép được với `"yx"`, `"xx"` chỉ ghép với `"xx"`. Ghép thêm một cặp luôn tăng 4, còn tâm chỉ đáng 2 ⇒ không bao giờ nên "phá" một cặp `"xx"` để lấy tâm (phá cặp mất 4, được 2… và thực ra phá cặp thì còn 2 bản lẻ, vẫn chỉ đặt 1 bản ở tâm).

**Đếm hai lần trong cách doocs:** duyệt mọi khóa `k` không đối xứng, cộng `min(v, cnt[k[::-1]]) * 2`. Mỗi cặp loại `("xy","yx")` được gặp **hai lần** (ở `"xy"` và ở `"yx"`), mỗi lần cộng `2·min` ⇒ tổng `4·min` — đúng.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Ghép cặp tham lam một lượt (ghép ngay khi gặp từ đảo)

Duyệt từng từ `w`: nếu có một bản **chưa ghép** của `reverse(w)` đang chờ thì ghép (+4) và giảm số chờ; ngược lại thêm `w` vào hàng chờ. Cuối cùng, nếu trong hàng chờ có từ đối xứng ⇒ +2.

Từ đối xứng `"gg"` cũng xử lý đúng: `reverse("gg") = "gg"`, bản thứ hai sẽ ghép với bản thứ nhất đang chờ.

In [8]:
def greedy_one_pass(words):
    waiting = Counter()
    ans = 0
    for w in words:
        r = w[::-1]
        if waiting[r] > 0:
            waiting[r] -= 1
            ans += 4
        else:
            waiting[w] += 1
    if any(w[0] == w[1] and c > 0 for w, c in waiting.items()):
        ans += 2
    return ans

for w, exp in examples:
    got = greedy_one_pass(w)
    print(got, "OK" if got == exp else "SAI")

6 OK
8 OK
2 OK


**Độ phức tạp cách 2:** O(n) thời gian (mỗi từ O(1)), O(số loại từ) ≤ 676 bộ nhớ, cộng bước cuối duyệt ≤ 676 khóa.

### 7.2 Cách 3 — `Counter` + công thức (cách doocs, viết rõ hơn)

In [9]:
def counter_formula(words):
    cnt = Counter(words)
    ans = 0
    has_center = False
    for w, v in cnt.items():
        if w[0] == w[1]:
            ans += (v // 2) * 4
            if v % 2:
                has_center = True
        elif w < w[::-1]:                  # xử lý mỗi cặp loại đúng một lần
            ans += min(v, cnt[w[::-1]]) * 4
    return ans + (2 if has_center else 0)

for w, exp in examples:
    got = counter_formula(w)
    print(got, "OK" if got == exp else "SAI")

6 OK
8 OK
2 OK


Ở đây dùng điều kiện `w < w[::-1]` (so sánh từ điển) để mỗi cặp loại `{"lc","cl"}` chỉ được tính một lần, cộng thẳng `4·min`. Lưu ý: nếu chỉ `"cl"` có trong `cnt` mà `"lc"` không có thì `min = 0` nên bỏ qua `"cl"` cũng không mất gì.

Code nộp LeetCode theo đúng kiểu doocs (đếm hai lần, mỗi lần `2·min`):

In [10]:
class SolutionCounter:
    def longestPalindrome(self, words: List[str]) -> int:
        cnt = Counter(words)
        ans = x = 0
        for k, v in cnt.items():
            if k[0] == k[1]:
                x += v & 1                 # đếm số loại đối xứng còn lẻ
                ans += v // 2 * 4
            else:
                ans += min(v, cnt[k[::-1]]) * 2
        return ans + (2 if x else 0)

print([SolutionCounter().longestPalindrome(w) for w, _ in examples])

[6, 8, 2]


Một chi tiết tinh tế: `cnt[k[::-1]]` trong lúc **đang duyệt** `cnt.items()`. Với `Counter`, truy cập khóa thiếu trả về 0 **mà không thêm khóa**, nên không gây lỗi "dict changed size during iteration". (Nếu dùng `defaultdict(int)` thay cho `Counter`, truy cập khóa thiếu sẽ **thêm** khóa ⇒ `RuntimeError`! Xem mục 13.)

**Độ phức tạp cách 3:** O(n) để đếm + O(676) để duyệt ⇒ **O(n)**, bộ nhớ O(676).

### 7.3 Cách 4 — Mảng 26 × 26

In [11]:
def grid26(words):
    g = [[0] * 26 for _ in range(26)]
    for w in words:
        g[ord(w[0]) - 97][ord(w[1]) - 97] += 1
    ans = 0; center = False
    for i in range(26):
        v = g[i][i]
        ans += (v // 2) * 4
        center |= v % 2 == 1
        for j in range(i + 1, 26):
            ans += min(g[i][j], g[j][i]) * 4
    return ans + 2 * center

for w, exp in examples:
    got = grid26(w)
    print(got, "OK" if got == exp else "SAI")

6 OK
8 OK
2 OK


Duyệt `j > i` để mỗi cặp ô `(i, j)` / `(j, i)` chỉ tính một lần; đường chéo `(i, i)` là từ đối xứng. `2 * center` với `center` là bool cho 0 hoặc 2.

**Độ phức tạp cách 4:** O(n + 26²), bộ nhớ O(26²) — cố định, không băm chuỗi.

## 8. TRACE từng bước

### 8.1 Trace cách 2 (ghép một lượt) trên ví dụ 2

| bước | từ | đảo | có bản đảo đang chờ? | hành động | ans | hàng chờ |
|---|---|---|---|---|---|---|
| 1 | ab | ba | không | chờ | 0 | {ab:1} |
| 2 | ty | yt | không | chờ | 0 | {ab:1, ty:1} |
| 3 | yt | ty | **có** | ghép | 4 | {ab:1, ty:0} |
| 4 | lc | cl | không | chờ | 4 | {…, lc:1} |
| 5 | cl | lc | **có** | ghép | 8 | {…, lc:0} |
| 6 | ab | ba | không | chờ | 8 | {ab:2, …} |

Không có từ đối xứng chờ ⇒ đáp án 8.

In [12]:
def trace_greedy(words):
    waiting = Counter(); ans = 0
    for step, w in enumerate(words, 1):
        r = w[::-1]
        if waiting[r] > 0:
            waiting[r] -= 1; ans += 4; act = f"ghép với '{r}' đang chờ"
        else:
            waiting[w] += 1; act = "đưa vào hàng chờ"
        print(f"bước {step}: '{w}' -> {act:<22} ans={ans:<3} chờ={ {k: v for k, v in waiting.items() if v} }")
    sym = [w for w, c in waiting.items() if c and w[0] == w[1]]
    print("từ đối xứng còn chờ:", sym, "->", "+2" if sym else "+0")
    return ans + (2 if sym else 0)

print("Kết quả:", trace_greedy(["ab", "ty", "yt", "lc", "cl", "ab"]))

bước 1: 'ab' -> đưa vào hàng chờ       ans=0   chờ={'ab': 1}
bước 2: 'ty' -> đưa vào hàng chờ       ans=0   chờ={'ab': 1, 'ty': 1}
bước 3: 'yt' -> ghép với 'ty' đang chờ ans=4   chờ={'ab': 1}
bước 4: 'lc' -> đưa vào hàng chờ       ans=4   chờ={'ab': 1, 'lc': 1}
bước 5: 'cl' -> ghép với 'lc' đang chờ ans=8   chờ={'ab': 1}
bước 6: 'ab' -> đưa vào hàng chờ       ans=8   chờ={'ab': 2}
từ đối xứng còn chờ: [] -> +0
Kết quả: 8


### 8.2 Trace cách 3 với nhiều từ đối xứng: `["gg","gg","gg","aa","ab","ba","ba"]`

In [13]:
def trace_counter(words):
    cnt = Counter(words); ans = 0; center = False
    print("cnt =", dict(cnt))
    for w, v in cnt.items():
        if w[0] == w[1]:
            ans += (v // 2) * 4; center |= v % 2 == 1
            print(f"  '{w}' đối xứng, v={v}: {v // 2} cặp (+{(v // 2) * 4}), dư {v % 2} -> ans={ans}, có tâm={center}")
        elif w < w[::-1]:
            k = min(v, cnt[w[::-1]]); ans += 4 * k
            print(f"  '{w}' & '{w[::-1]}': min({v}, {cnt[w[::-1]]}) = {k} cặp (+{4 * k}) -> ans={ans}")
        else:
            print(f"  '{w}': đã tính cùng '{w[::-1]}'")
    print("Kết quả:", ans + (2 if center else 0))

trace_counter(["gg", "gg", "gg", "aa", "ab", "ba", "ba"])

cnt = {'gg': 3, 'aa': 1, 'ab': 1, 'ba': 2}
  'gg' đối xứng, v=3: 1 cặp (+4), dư 1 -> ans=4, có tâm=True
  'aa' đối xứng, v=1: 0 cặp (+0), dư 1 -> ans=4, có tâm=True
  'ab' & 'ba': min(1, 2) = 1 cặp (+4) -> ans=8
  'ba': đã tính cùng 'ab'
Kết quả: 10


## 9. Code nộp LeetCode

In [14]:
from typing import List
from collections import Counter

class Solution:
    def longestPalindrome(self, words: List[str]) -> int:
        cnt = Counter(words)
        ans = 0
        center = False
        for w, v in cnt.items():
            rw = w[::-1]
            if w == rw:                          # "aa": ghép với chính loại của nó
                ans += (v // 2) * 4
                if v % 2 == 1:
                    center = True
            elif w < rw:                         # mỗi cặp loại tính một lần
                ans += min(v, cnt[rw]) * 4
        if center:
            ans += 2                             # đúng MỘT từ đối xứng ở tâm
        return ans

In [15]:
sol = Solution()
for w, exp in examples:
    got = sol.longestPalindrome(w)
    print(got, "| kỳ vọng", exp, "| OK" if got == exp else "| SAI")

6 | kỳ vọng 6 | OK
8 | kỳ vọng 8 | OK
2 | kỳ vọng 2 | OK


## 10. Vì sao thuật toán đúng

**Cận trên.** Xét palindrome tốt nhất gồm `k` khối. Theo mục 4, các khối ghép thành `⌊k/2⌋` cặp `(B_i, B_{k+1−i})` với `B_{k+1−i} = reverse(B_i)`, cộng tối đa một khối giữa đối xứng.
- Số cặp dùng loại `{"xy","yx"}` (x ≠ y) không vượt `min(cnt["xy"], cnt["yx"])` vì mỗi cặp dùng một bản mỗi loại.
- Số cặp dùng loại `"xx"` không vượt `cnt["xx"] // 2`.
- Khối giữa (nếu có) là một bản `"xx"`. Nếu `cnt["xx"]` **lẻ**: dùng 1 bản làm tâm vẫn còn `cnt − 1` (chẵn) bản cho `cnt // 2` cặp — không mất gì. Nếu `cnt["xx"]` **chẵn**: lấy 1 bản làm tâm thì loại này chỉ còn tối đa `cnt/2 − 1` cặp ⇒ tổng ≤ `4P − 4 + 2 < 4P`, tệ hơn không đặt tâm.
- Vậy độ dài ≤ `4P + 2·[tồn tại loại đối xứng có số lượng lẻ]`, với `P` là tổng các cận trên về số cặp.

**Đạt được.** Thuật toán xây đúng `P` cặp (đặt cặp thứ `i` ở vị trí `i` và `k+1−i`) và đặt một từ đối xứng lẻ ở giữa nếu có ⇒ chuỗi là palindrome với độ dài đúng bằng cận trên. ∎

**Cách 2 (một lượt) cũng tối ưu:** với mỗi loại cặp, số lần "ghép" ở cách 2 bằng `min(cnt["xy"], cnt["yx"])` (hàng chờ chỉ chứa loại đang dư), và với `"xx"` bằng `cnt // 2`; cuối cùng hàng chờ chứa `"xx"` ⇔ `cnt["xx"]` lẻ.

## 11. Phân tích Time / Space Complexity

`n = len(words)`; số loại từ khác nhau ≤ 26² = 676.

| Cách | Time | Space |
|---|---|---|
| 1. Tập con + hoán vị | O(n!·n) | O(n) |
| 2. Ghép một lượt | O(n) | O(676) |
| 3. Counter + công thức | O(n) | O(676) |
| 4. Mảng 26×26 | O(n + 676) | O(676) |

In [16]:
rng = random.Random(2131)
letters = "abcdefghijklmnopqrstuvwxyz"
big = ["".join(rng.choice(letters[:6]) for _ in range(2)) for _ in range(10**5)]
for name, f in [("greedy_one_pass", greedy_one_pass), ("counter_formula", counter_formula),
                ("SolutionCounter", SolutionCounter().longestPalindrome), ("grid26", grid26),
                ("Solution", Solution().longestPalindrome)]:
    t = time.perf_counter(); r = f(big); dt = time.perf_counter() - t
    print(f"{name:<16} -> {r}  {dt*1000:.1f} ms")

greedy_one_pass  -> 197942  28.8 ms


counter_formula  -> 197942  5.6 ms
SolutionCounter  -> 197942  3.7 ms
grid26           -> 197942  5.7 ms
Solution         -> 197942  3.8 ms


## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [17]:
ALL = {"brute": brute, "greedy_one_pass": greedy_one_pass, "counter_formula": counter_formula,
       "SolutionCounter": SolutionCounter().longestPalindrome, "grid26": grid26,
       "Solution": Solution().longestPalindrome}
edge = examples + [
    (["ab"], 0),
    (["aa"], 2),
    (["ab", "ba"], 4),
    (["aa", "aa"], 4),
    (["aa", "aa", "aa"], 6),            # cặp + tâm
    (["aa", "bb", "aa", "bb"], 8),       # 2 cặp, không tâm
    (["aa", "bb", "cc", "aa"], 6),       # 1 cặp + 1 tâm (bb hoặc cc)
    (["ab", "ab", "ba"], 4),
    (["ab", "ba", "ba", "ab", "ab"], 8),
    (["em", "pe", "mp", "ee", "pp", "me", "ep", "em", "em", "me"], 14),
]
for w, exp in edge:
    for name, f in ALL.items():
        if name == "brute" and len(w) > 7: continue
        assert f(list(w)) == exp, (name, w, f(list(w)), exp)
print("Edge case OK:", len(edge))

Edge case OK: 13


Case cuối `["em","pe","mp","ee","pp","me","ep","em","em","me"]` tự dựng (đối chiếu lại đề gốc nếu muốn): cặp `em/me` × 2 (dư một `"em"`), `pe/ep` × 1, `mp` không có `"pm"`, `ee`, `pp` là hai từ đối xứng lẻ ⇒ `4·3 + 2 = 14`. Nó quá lớn cho brute force (10 từ), nên ta tự kiểm bằng cách **dựng chuỗi** và kiểm tra palindrome ở mục 12.3.

### 12.2 Đối chiếu ngẫu nhiên với brute force (n ≤ 6, bảng chữ nhỏ để hay có cặp)

In [18]:
rng = random.Random(42)
for _ in range(1200):
    n = rng.randint(1, 6)
    alpha = rng.choice(["ab", "abc", "abcd"])
    w = ["".join(rng.choice(alpha) for _ in range(2)) for _ in range(n)]
    ref = brute(w)
    for name, f in ALL.items():
        assert f(list(w)) == ref, (name, w)
print("1200 case ngẫu nhiên OK")

1200 case ngẫu nhiên OK


### 12.3 Kiểm tra bằng cách **dựng palindrome thật** (input lớn)

Dựng chuỗi theo mục 4, kiểm tra: (1) là palindrome, (2) độ dài bằng đáp án, (3) mỗi từ dùng không quá số lần có trong input.

In [19]:
def construct(words):
    cnt = Counter(words); pairs = []; center = ""
    for w, v in cnt.items():
        rw = w[::-1]
        if w == rw:
            pairs += [w] * (v // 2)
            if v % 2 and not center: center = w
        elif w < rw:
            pairs += [w] * min(v, cnt[rw])
    left = "".join(pairs); right = "".join(w[::-1] for w in reversed(pairs))
    used = Counter(pairs) + Counter(w[::-1] for w in pairs) + (Counter([center]) if center else Counter())
    return left + center + right, used

rng = random.Random(7)
for _ in range(300):
    w = ["".join(rng.choice("abcde") for _ in range(2)) for _ in range(rng.randint(1, 500))]
    s, used = construct(w)
    assert s == s[::-1] and len(s) == Solution().longestPalindrome(w)
    assert all(used[k] <= Counter(w)[k] for k in used)
print("dựng palindrome thật: OK")

dựng palindrome thật: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cộng +2 cho **mọi** từ đối xứng lẻ (nhiều tâm)

In [20]:
def wrong_many_centers(words):
    cnt = Counter(words); ans = 0
    for w, v in cnt.items():
        if w[0] == w[1]:
            ans += (v // 2) * 4 + (v % 2) * 2          # SAI: mỗi loại lẻ đều +2
        elif w < w[::-1]:
            ans += min(v, cnt[w[::-1]]) * 4
    return ans

print("bản sai :", wrong_many_centers(["cc", "ll", "xx"]))
print("bản đúng:", Solution().longestPalindrome(["cc", "ll", "xx"]))

bản sai : 6
bản đúng: 2


**Vì sao sai:** palindrome chỉ có **một** tâm. `"ccllxx"` không phải palindrome. **Sửa:** dùng cờ `center` và cộng 2 **một lần** ở cuối.

### Lỗi 2 — Đếm cặp không đối xứng hai lần (mỗi lần `4·min`)

In [21]:
def wrong_double_count(words):
    cnt = Counter(words); ans = 0; center = False
    for w, v in cnt.items():
        if w[0] == w[1]:
            ans += (v // 2) * 4; center |= v % 2 == 1
        else:
            ans += min(v, cnt[w[::-1]]) * 4             # SAI: gặp cả "lc" lẫn "cl"
    return ans + 2 * center

print("bản sai :", wrong_double_count(["lc", "cl", "gg"]))
print("bản đúng:", Solution().longestPalindrome(["lc", "cl", "gg"]))

bản sai : 10
bản đúng: 6


**Sửa:** hoặc `* 2` khi duyệt cả hai chiều (như doocs), hoặc chỉ xét `w < reverse(w)` và `* 4`.

### Lỗi 3 — Dùng `defaultdict(int)` rồi truy cập khóa đảo trong lúc duyệt

In [22]:
from collections import defaultdict
def wrong_defaultdict(words):
    cnt = defaultdict(int)
    for w in words: cnt[w] += 1
    ans = 0; center = False
    try:
        for w, v in cnt.items():
            if w[0] == w[1]:
                ans += (v // 2) * 4; center |= v % 2 == 1
            else:
                ans += min(v, cnt[w[::-1]]) * 2      # tạo khóa mới khi "ba" chưa có!
        return ans + 2 * center
    except RuntimeError as e:
        return f"RuntimeError: {e}"

print("bản sai :", wrong_defaultdict(["ab", "ty", "yt", "lc", "cl", "ab"]))
print("bản đúng:", Solution().longestPalindrome(["ab", "ty", "yt", "lc", "cl", "ab"]))

bản sai : RuntimeError: dictionary changed size during iteration
bản đúng: 8


**Vì sao sai:** `defaultdict.__getitem__` với khóa thiếu sẽ **chèn** khóa đó ⇒ dict đổi kích thước trong lúc duyệt ⇒ `RuntimeError`. **Sửa:** dùng `Counter` (không chèn), hoặc `cnt.get(rw, 0)`, hoặc duyệt `list(cnt.items())`.

### Lỗi 4 — Nghĩ rằng từ đối xứng chỉ dùng được ở tâm

In [23]:
def wrong_sym_only_center(words):
    cnt = Counter(words); ans = 0; center = False
    for w, v in cnt.items():
        if w[0] == w[1]:
            center = True                                # SAI: bỏ quên cặp "gg"+"gg"
        elif w < w[::-1]:
            ans += min(v, cnt[w[::-1]]) * 4
    return ans + 2 * center

print("bản sai :", wrong_sym_only_center(["gg", "gg", "ab", "ba"]))
print("bản đúng:", Solution().longestPalindrome(["gg", "gg", "ab", "ba"]))

bản sai : 6
bản đúng: 8


**Sửa:** hai bản `"gg"` đặt đối xứng hai bên như một cặp bình thường (`"gg…gg"`), đóng góp 4.

### Lỗi 5 — Dùng `max` thay vì `min` khi ghép cặp

In [24]:
def wrong_max(words):
    cnt = Counter(words); ans = 0; center = False
    for w, v in cnt.items():
        if w[0] == w[1]:
            ans += (v // 2) * 4; center |= v % 2 == 1
        elif w < w[::-1] and cnt[w[::-1]]:
            ans += max(v, cnt[w[::-1]]) * 4             # SAI
    return ans + 2 * center

print("bản sai :", wrong_max(["ab", "ab", "ba"]))
print("bản đúng:", Solution().longestPalindrome(["ab", "ab", "ba"]))

bản sai : 8
bản đúng: 4


**Sửa:** mỗi cặp cần **một bản của mỗi loại** ⇒ số cặp bị giới hạn bởi loại **ít** hơn: `min`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Ghép thành palindrome, thứ tự tùy ý" ⇒ đừng nghĩ về thứ tự, nghĩ về **tần suất** và **cặp đối xứng**. Giống LC 409 Longest Palindrome (ghép ký tự).
- Palindrome: **hai bên đối xứng** + **tối đa một tâm**.
- Đếm cặp với `Counter`; cẩn thận đếm đôi khi duyệt cả `w` và `reverse(w)`.
- Bảng chữ nhỏ (26²) ⇒ mảng 2 chiều thay `dict` khi cần tốc độ / ngôn ngữ khác.
- Bài cùng họ: 409, 336 Palindrome Pairs (khó hơn, độ dài tùy ý), 1400 Construct K Palindrome Strings, 2193.

## 15. Biến thể của bài

### 15.1 Biến thể A — Trả về **chính chuỗi palindrome** nhỏ nhất theo thứ tự từ điển (trong các chuỗi dài nhất)

Để chuỗi nhỏ nhất: đặt các cặp sao cho nửa trái tăng dần theo từ điển (sắp xếp các từ bên trái), chọn từ trái của mỗi cặp là bản **nhỏ hơn** giữa `w` và `reverse(w)`, và tâm là từ đối xứng lẻ **nhỏ nhất**. (Với mọi cặp, nửa trái dùng `min(w, rw)` là tối ưu vì chuỗi được so sánh từ trái sang.)

In [25]:
def smallest_longest_palindrome(words):
    cnt = Counter(words); left = []; centers = []
    for w, v in cnt.items():
        rw = w[::-1]
        if w == rw:
            left += [w] * (v // 2)
            if v % 2: centers.append(w)
        elif w < rw:
            left += [w] * min(v, cnt[rw])
    left.sort()
    mid = min(centers) if centers else ""
    return "".join(left) + mid + "".join(w[::-1] for w in reversed(left))

def smallest_brute(words):
    best = None; n = len(words)
    for k in range(n, 0, -1):
        for comb in combinations(range(n), k):
            for perm in permutations(comb):
                s = "".join(words[i] for i in perm)
                if s == s[::-1] and (best is None or s < best): best = s
        if best is not None: return best
    return ""

for w in [["lc", "cl", "gg"], ["cc", "ll", "xx"], ["ba", "ab", "dd", "aa", "aa"]]:
    print(w, "->", repr(smallest_longest_palindrome(w)))
rng = random.Random(151)
for _ in range(300):
    w = ["".join(rng.choice("abc") for _ in range(2)) for _ in range(rng.randint(1, 5))]
    assert smallest_longest_palindrome(w) == smallest_brute(w), w
print("khớp brute force")

['lc', 'cl', 'gg'] -> 'clgglc'
['cc', 'll', 'xx'] -> 'cc'
['ba', 'ab', 'dd', 'aa', 'aa'] -> 'aaabddbaaa'
khớp brute force


### 15.2 Biến thể B — Các từ dài `L` chữ (không chỉ 2)

Với độ dài cố định `L`, lập luận "khối đối xứng qua tâm" vẫn đúng y nguyên: `B_i = reverse(B_{k+1−i})`, tâm là từ tự đối xứng. Code giữ nguyên, chỉ thay điều kiện `w[0] == w[1]` bằng `w == w[::-1]` và nhân `2L` thay cho 4.

In [26]:
def longest_palindrome_len_L(words):
    L = len(words[0]); cnt = Counter(words); pairs = 0; center = False
    for w, v in cnt.items():
        rw = w[::-1]
        if w == rw:
            pairs += v // 2; center |= v % 2 == 1
        elif w < rw:
            pairs += min(v, cnt[rw])
    return pairs * 2 * L + (L if center else 0)

print(longest_palindrome_len_L(["abc", "cba", "aba", "xyz"]))      # abc aba cba -> 9
print(longest_palindrome_len_L(["lc", "cl", "gg"]))                # = bài gốc: 6
rng = random.Random(152)
for _ in range(300):
    w = ["".join(rng.choice("ab") for _ in range(3)) for _ in range(rng.randint(1, 5))]
    assert longest_palindrome_len_L(w) == brute(w) // 2 * 3
print("khớp brute force với L = 3")

9
6
khớp brute force với L = 3


### 15.3 Biến thể C — Mỗi từ có thể dùng **không giới hạn** số lần

Nếu có cả `"xy"` và `"yx"` (hoặc bất kỳ `"xx"`) thì palindrome dài vô hạn ⇒ trả `-1` (vô hạn); ngược lại 0. Biến thể này cho thấy giới hạn "mỗi phần tử một lần" chính là thứ làm bài toán thú vị.

In [27]:
def unlimited(words):
    s = set(words)
    return -1 if any(w[::-1] in s for w in s) else 0

print(unlimited(["ab", "cd"]), unlimited(["ab", "ba"]), unlimited(["gg"]))

0 -1 -1


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | mọi tập con + hoán vị | O(n!·n) | O(n) | kiểm chứng n ≤ 6 |
| 2. Ghép một lượt | hàng chờ, gặp bản đảo thì ghép | O(n) | O(676) | streaming, trực quan |
| 3. Counter + công thức | `4·min`, `4·(v//2)`, tâm +2 | O(n) | O(676) | **mặc định**, ngắn |
| 4. Mảng 26×26 | như 3, không băm | O(n + 676) | O(676) | tốc độ, C++/Java |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `longestPalindrome(["aa","ab","ba","aa","aa"])` và `longestPalindrome(["xy","yz","zx"])`?

**Bài 2 (tính tay):** Với `["ab","ba","ab","ba","ab","cc","cc","dd"]`, có bao nhiêu cặp, có tâm không, độ dài bao nhiêu?

**Bài 3 (điền TODO):** Viết `bt3_todo(words)` trả về **số từ không được dùng** trong một palindrome dài nhất.

**Bài 4 (tự viết lại):** Viết `bt4_todo(words)` cài cách 4 (mảng 26×26) không nhìn code.

**Bài 5 (mini challenge):** Viết `bt5_todo(words)` trả về **một** palindrome dài nhất (chuỗi bất kỳ, `""` nếu không có) — kiểm tra bằng cách xác nhận nó là palindrome, độ dài đúng, và chỉ dùng từ có trong input.

In [28]:
def bt3_todo(words):
    # TODO: số từ không dùng tới
    pass

def bt4_todo(words):
    # TODO: mảng 26x26
    pass

def bt5_todo(words):
    # TODO: dựng một palindrome dài nhất
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: đếm. Mức 2: `aa` × 3 ⇒ 1 cặp + tâm; `ab/ba` 1 cặp. Mức 3: `4 + 4 + 2 = 10`; ví dụ hai: không có cặp nào, không có từ đối xứng ⇒ `0`.

**Bài 2.** Mức 1: `ab` × 3, `ba` × 2 ⇒ `min = 2` cặp. Mức 2: `cc` × 2 ⇒ 1 cặp; `dd` × 1 ⇒ tâm. Mức 3: `4·3 + 2 = 14`.

**Bài 3.** Mức 1: số từ dùng = độ dài / 2. Mức 2: `len(words) − ans // 2`. Mức 3: đúng vậy — mỗi từ 2 ký tự.

**Bài 4.** Mức 1: `g[x][y] += 1`. Mức 2: đường chéo xử lý riêng. Mức 3: duyệt `j > i` với `min(g[i][j], g[j][i])`.

**Bài 5.** Mức 1: tái dùng `construct` ở mục 12.3. Mức 2: nửa trái = danh sách từ trái của mỗi cặp. Mức 3: nửa phải = đảo ngược thứ tự và đảo từng từ.

## 19. Đáp án tham khảo cho bài tập

**Bài 1:** `10`, `0`. **Bài 2:** 3 cặp (`ab/ba` × 2, `cc/cc` × 1), có tâm (`dd`), độ dài `14`.

In [29]:
assert Solution().longestPalindrome(["aa", "ab", "ba", "aa", "aa"]) == 10
assert Solution().longestPalindrome(["xy", "yz", "zx"]) == 0
assert Solution().longestPalindrome(["ab", "ba", "ab", "ba", "ab", "cc", "cc", "dd"]) == 14

def bt3(words):
    return len(words) - Solution().longestPalindrome(words) // 2

def bt4(words):
    g = [[0] * 26 for _ in range(26)]
    for w in words: g[ord(w[0]) - 97][ord(w[1]) - 97] += 1
    ans = 0; center = False
    for i in range(26):
        ans += g[i][i] // 2 * 4
        if g[i][i] % 2: center = True
        for j in range(i + 1, 26):
            ans += 4 * min(g[i][j], g[j][i])
    return ans + (2 if center else 0)

def bt5(words):
    return construct(words)[0]

assert bt3(["ab", "ty", "yt", "lc", "cl", "ab"]) == 2 and bt3(["cc", "ll", "xx"]) == 2
rng = random.Random(1931)
for _ in range(1500):
    w = ["".join(rng.choice("abcd") for _ in range(2)) for _ in range(rng.randint(1, 6))]
    ref = brute(w)
    assert bt4(w) == ref
    assert bt3(w) == len(w) - ref // 2
    s = bt5(w)
    assert s == s[::-1] and len(s) == ref
    blocks = Counter(s[i:i + 2] for i in range(0, len(s), 2))
    assert all(blocks[k] <= Counter(w)[k] for k in blocks)
print("Đáp án bài tập: tất cả assert OK")

Đáp án bài tập: tất cả assert OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cnt = Counter(words); ans = 0; center = False
for w, v in cnt.items():
    rw = w[::-1]
    if w == rw:
        ans += v // 2 * 4
        center |= v % 2 == 1
    elif w < rw:
        ans += min(v, cnt[rw]) * 4
return ans + (2 if center else 0)
```

**Checklist:**
- [ ] Tâm chỉ cộng **một lần** (+2).
- [ ] Cặp không đối xứng không bị đếm đôi (`w < rw` với `*4`, hoặc duyệt hết với `*2`).
- [ ] Từ đối xứng cũng ghép cặp được (`v // 2`).
- [ ] Dùng `Counter` (không chèn khóa khi đọc) hoặc `.get`.
- [ ] Đã thử: `["cc","ll","xx"]`, `["aa","aa","aa"]`, `["ab","ab","ba"]`.